# 05 — XGBoost valuation model (real Dubizzle UAE listings)

Trains the price model on **672 real, deduplicated Dubizzle UAE used-car listings** scraped July 2026
(`data/prepare_tabular.py` output; see DECISIONS.md ADR-011 for why the synthetic Kaggle set was rejected).

**Design**
- Target: `log_price` (right-skew). Quantile regression at **α = 0.10 / 0.50 / 0.90**; exponentiating the
  predicted log-quantiles yields the low/mid/high **price range** directly. The q10–q90 spread is the
  prediction interval required by the confidence-disclosure contract (Section 15).
- No `condition` feature: the tabular model prices a *market-condition* vehicle; the CV damage detector's
  Condition Score is applied as a separate adjustment at inference. Clean separation of the two models.
- **Honest metrics**: 5-fold CV (MAE/RMSE/MAPE/median-APE in AED) on held-out folds only, plus empirical
  q10–q90 coverage, and a naive make+model-median baseline it must beat.

**Small-data caveat (stated honestly):** 672 rows is a modest set; error bars are wider than a
production model's and are disclosed, never hidden.

In [1]:
import json, joblib
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold

df = pd.read_parquet("../data/processed/listings_clean.parquet")
print("rows:", len(df))

CAT = ["make","model","bodyType","transmissionType","fuelType","regionalSpecs","sellerType","city"]
NUM = ["year","age","kilometers","mileage_per_year","noOfCylinders"]
FEATURES = CAT + NUM
TARGET = "log_price"

cat_maps = {}
X = df[FEATURES].copy()
for c in CAT:
    cats = sorted(df[c].astype(str).unique())
    cat_maps[c] = {v:i for i,v in enumerate(cats)}
    X[c] = df[c].astype(str).map(cat_maps[c]).astype("int32")
X[NUM] = X[NUM].astype("float32")   # noOfCylinders NaN stays NaN -> XGBoost handles natively
y = df[TARGET].values
price = df["price"].values
print("features:", X.shape)

rows: 672
features: (672, 13)


## 5-fold CV — held-out metrics only

In [2]:
PARAMS = dict(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=5,
    reg_lambda=2.0, random_state=42, n_jobs=-1,
    objective="reg:quantileerror",
)
QUANTILES = {"q10":0.10, "q50":0.50, "q90":0.90}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for fold,(tr,te) in enumerate(kf.split(X)):
    preds = {}
    for name,q in QUANTILES.items():
        m = xgb.XGBRegressor(**{**PARAMS, "quantile_alpha":q})
        m.fit(X.iloc[tr], y[tr], verbose=False)
        preds[name] = np.expm1(m.predict(X.iloc[te]))
    lo = np.minimum(preds["q10"], preds["q90"]); hi = np.maximum(preds["q10"], preds["q90"])
    true = price[te]; mid = preds["q50"]
    rows.append({
        "MAE_AED": float(np.mean(np.abs(mid-true))),
        "RMSE_AED": float(np.sqrt(np.mean((mid-true)**2))),
        "MAPE_pct": float(np.mean(np.abs(mid-true)/true)*100),
        "median_APE_pct": float(np.median(np.abs(mid-true)/true)*100),
        "coverage_q10_q90": float(np.mean((true>=lo)&(true<=hi))),
    })
    print(f"fold {fold}: {rows[-1]}")

cv = pd.DataFrame(rows)
summary = {k:{"mean":round(float(cv[k].mean()),2),"std":round(float(cv[k].std()),2)} for k in cv.columns}
print(json.dumps(summary, indent=2))

fold 0: {'MAE_AED': 54120.022135416664, 'RMSE_AED': 142184.14218816892, 'MAPE_pct': 28.400470174931108, 'median_APE_pct': 19.650586691602317, 'coverage_q10_q90': 0.6296296296296297}


fold 1: {'MAE_AED': 40210.27307581018, 'RMSE_AED': 125321.20668922996, 'MAPE_pct': 27.619493316038557, 'median_APE_pct': 17.399057262569833, 'coverage_q10_q90': 0.6222222222222222}


fold 2: {'MAE_AED': 43040.009554279386, 'RMSE_AED': 182558.07074448923, 'MAPE_pct': 27.552670442820364, 'median_APE_pct': 21.794700802351695, 'coverage_q10_q90': 0.5373134328358209}


fold 3: {'MAE_AED': 31956.797975454756, 'RMSE_AED': 92736.08359839459, 'MAPE_pct': 27.23875745819923, 'median_APE_pct': 19.98792956618789, 'coverage_q10_q90': 0.5671641791044776}


fold 4: {'MAE_AED': 29259.171073344216, 'RMSE_AED': 50407.40954565451, 'MAPE_pct': 26.97718106246072, 'median_APE_pct': 19.025245670180723, 'coverage_q10_q90': 0.582089552238806}
{
  "MAE_AED": {
    "mean": 39717.25,
    "std": 9852.91
  },
  "RMSE_AED": {
    "mean": 118641.38,
    "std": 50027.95
  },
  "MAPE_pct": {
    "mean": 27.56,
    "std": 0.54
  },
  "median_APE_pct": {
    "mean": 19.57,
    "std": 1.59
  },
  "coverage_q10_q90": {
    "mean": 0.59,
    "std": 0.04
  }
}


### Naive baseline — must beat make+model median

In [3]:
base = []
for tr,te in kf.split(X):
    a,b = df.iloc[tr], df.iloc[te]
    mm = a.groupby(["make","model"])["price"].median()
    mk = a.groupby("make")["price"].median()
    g = a["price"].median()
    def pred(r):
        return mm.get((r["make"],r["model"]), mk.get(r["make"], g))
    p = b.apply(pred, axis=1).values
    base.append(float(np.mean(np.abs(p-b["price"].values))))
baseline_mae = float(np.mean(base))
model_mae = summary["MAE_AED"]["mean"]
print(f"naive make+model median MAE: {baseline_mae:,.0f} AED")
print(f"XGBoost q50 MAE:             {model_mae:,.0f} AED")
print(f"improvement over baseline:   {100*(1-model_mae/baseline_mae):.1f}%")

naive make+model median MAE: 55,483 AED
XGBoost q50 MAE:             39,717 AED
improvement over baseline:   28.4%


## Fit final models on all data, export bundle

## Conformal interval calibration (honest 80% coverage)
Quantile regression under-covers on 672 rows (empirical ~0.59 vs nominal 0.80). We calibrate a
**split-conformal** interval: using out-of-fold q50 residuals in log space, pick the symmetric
log-offset `delta` whose |residual| quantile gives 80% coverage by construction. Applied at
inference as `[expm1(logq50 - delta), expm1(logq50 + delta)]` — a calibrated, defensible interval.

In [4]:
# out-of-fold q50 log-residuals
oof_log_resid = np.full(len(X), np.nan)
for tr,te in kf.split(X):
    m = xgb.XGBRegressor(**{**PARAMS,"quantile_alpha":0.50})
    m.fit(X.iloc[tr], y[tr], verbose=False)
    oof_log_resid[te] = y[te] - m.predict(X.iloc[te])   # log-space residual

abs_res = np.abs(oof_log_resid[~np.isnan(oof_log_resid)])
conformal_delta = float(np.quantile(abs_res, 0.80))     # 80% coverage target
# verify: q50 log-prediction = y - residual; bounds in price space vs true AED price
q50_log_pred = y - oof_log_resid
lo = np.expm1(q50_log_pred - conformal_delta)
hi = np.expm1(q50_log_pred + conformal_delta)
cov = float(np.mean((price >= lo) & (price <= hi)))
print(f"conformal_delta (log): {conformal_delta:.4f}")
print(f"calibrated interval empirical coverage: {cov:.3f} (target 0.80)")
print(f"typical band: mid x{np.exp(conformal_delta):.2f} (high) / ÷{np.exp(conformal_delta):.2f} (low)")

conformal_delta (log): 0.4158
calibrated interval empirical coverage: 0.799 (target 0.80)
typical band: mid x1.52 (high) / ÷1.52 (low)


In [5]:
final = {}
for name,q in QUANTILES.items():
    m = xgb.XGBRegressor(**{**PARAMS,"quantile_alpha":q})
    m.fit(X, y, verbose=False)
    final[name] = m

bundle = {
    "models": final, "cat_maps": cat_maps, "features": FEATURES,
    "categorical_features": CAT, "numeric_features": NUM,
    "target": "log_price (expm1 -> AED)", "quantiles": QUANTILES,
    "cv_metrics": summary, "baseline_mae_aed": round(baseline_mae,2),
    "training_rows": int(len(df)),
    "dataset": "Dubizzle UAE scrape, July 2026 (672 real listings)",
    "reference_year": 2026,
    "conformal_delta_log": conformal_delta,
    "conformal_coverage": round(float(cov),3),
}
joblib.dump(bundle, "../backend-api/models/valuation_model.joblib", compress=3)

metrics_out = {
    "model": "XGBoost quantile regression (q10/q50/q90) on log_price",
    "cv": "5-fold shuffled, seed 42 — held-out folds only",
    "metrics": summary,
    "baseline_make_model_median_MAE_AED": round(baseline_mae,2),
    "improvement_over_baseline_pct": round(100*(1-model_mae/baseline_mae),1),
    "training_rows": int(len(df)),
    "calibrated_interval_coverage": round(float(cov),3),
    "conformal_delta_log": round(conformal_delta,4),
    "dataset": "Dubizzle UAE scrape July 2026 (real)",
}
json.dump(metrics_out, open("../eval/valuation_metrics.json","w"), indent=2)
import os
print("bundle:", round(os.path.getsize('../backend-api/models/valuation_model.joblib')/1e6,2),"MB")
print(json.dumps(metrics_out, indent=2))

bundle: 0.58 MB
{
  "model": "XGBoost quantile regression (q10/q50/q90) on log_price",
  "cv": "5-fold shuffled, seed 42 \u2014 held-out folds only",
  "metrics": {
    "MAE_AED": {
      "mean": 39717.25,
      "std": 9852.91
    },
    "RMSE_AED": {
      "mean": 118641.38,
      "std": 50027.95
    },
    "MAPE_pct": {
      "mean": 27.56,
      "std": 0.54
    },
    "median_APE_pct": {
      "mean": 19.57,
      "std": 1.59
    },
    "coverage_q10_q90": {
      "mean": 0.59,
      "std": 0.04
    }
  },
  "baseline_make_model_median_MAE_AED": 55483.36,
  "improvement_over_baseline_pct": 28.4,
  "training_rows": 672,
  "calibrated_interval_coverage": 0.799,
  "conformal_delta_log": 0.4158,
  "dataset": "Dubizzle UAE scrape July 2026 (real)"
}
